# Euroleague Data Pipeline

Pulls player box scores + team metadata from the `euroleague_api` package and builds `boxscores_final.csv` — the file the Streamlit apps (`app.py`, `app_props.py`) read.

Data has already been pulled through the end of the 2025-26 season. You only need to re-run this if you want to refresh existing seasons or add a new one. The pipeline is: **team metadata -> player headshots -> per-season boxscores -> combine seasons -> enrich with team/matchup info**.

In [ ]:
from euroleague_api.boxscore_data import BoxScoreData
from euroleague_api.schedule import Schedule
from euroleague_api.team_stats import TeamStats
from euroleague_api.player_stats import PlayerStats
import pandas as pd
import numpy as np
import time

pd.set_option('display.max_columns', None)

COMPETITION_CODE = "E"   # Euroleague
SEASONS = [2024, 2025]   # season = year the season *starts* in (season 2025 = 2025-26 season)

## 1. Team metadata

Team codes, names, and logo URLs — used to attach a team name/logo to every player-game row.

In [ ]:
team_stats_api = TeamStats(competition=COMPETITION_CODE)
teams = team_stats_api.get_team_stats(endpoint='traditional')
teams = teams[['team.code', 'team.name', 'team.imageUrl']]
teams.to_csv('teams.csv', index=False)
teams.head()

## 2. Player headshots -> `players.csv`

The `PlayerStats.traditional` endpoint returns a `player.imageUrl` headshot alongside per-player stats. Note: with the default `statistic_mode="PerGame"` the API silently drops fringe players below some games-played threshold (confirmed: it excludes real rotation players with 15-25 games played, not just garbage-time call-ups). Using `statistic_mode="Accumulated"` instead recovers most of them — this covers 439 of the ~470 players who appear in the actual boxscores; the remainder are one-appearance emergency signings not in the League's official stats at all, and simply won't have a photo (the apps fall back to the team logo).

Only run this when you actually want to refresh headshots (new season, transfers, etc.) — `players.csv` already exists, and the apps just read it, no API calls happen when you browse players in Streamlit.

In [ ]:
player_stats_api = PlayerStats(competition=COMPETITION_CODE)

player_frames = []
for season in SEASONS:
    season_players = player_stats_api.get_player_stats_single_season(
        'traditional', season=season, statistic_mode='Accumulated'
    )
    player_frames.append(season_players[['player.code', 'player.name', 'player.imageUrl']])

# keep='last' -> a player who appears in multiple seasons gets their most recent photo
players = pd.concat(player_frames, ignore_index=True).drop_duplicates(subset=['player.code'], keep='last')
players.columns = ['PlayerCode', 'Player', 'PlayerImageUrl']
players.to_csv('players.csv', index=False)
players.shape

## 3. Boxscores + schedule, per season

For each season: pull the schedule (for game date / home & away teams) and every game's player box scores, then merge them. Saves one CSV per season (`boxscores_{season}.csv`) so an interrupted run doesn't lose already-fetched seasons.

This hits the API once per game in the season (a few hundred requests), so it's slow — that's what the `time.sleep` is for.

In [ ]:
def fetch_season_boxscores(season: int, competition: str = COMPETITION_CODE, request_delay: float = 1.0) -> pd.DataFrame:
    """Fetch every player's box score for a season, merged with schedule info (date, home/away team)."""
    schedule_api = Schedule(competition=competition)
    boxscore_api = BoxScoreData(competition=competition)

    schedule = schedule_api.get_schedule(season=season)
    schedule['game'] = schedule['game'].astype(int)
    schedule['date'] = pd.to_datetime(schedule['date'])

    gamecodes = boxscore_api.get_gamecodes_season(season=season)

    boxscores = pd.DataFrame()
    for code in gamecodes['gameCode']:
        game_df = boxscore_api.get_players_boxscore_stats(season=season, gamecode=code)
        boxscores = pd.concat([boxscores, game_df], ignore_index=True)
        time.sleep(request_delay)

    schedule_cols = ['game', 'date', 'startime', 'endtime', 'group', 'hometeam', 'awayteam']
    boxscores_merged = boxscores.merge(
        schedule[schedule_cols], how='left', left_on='Gamecode', right_on='game'
    ).drop(columns=['game'])

    return boxscores_merged

In [ ]:
# Re-run for a single season by narrowing SEASONS, e.g. SEASONS = [2026] once the new season starts.
for season in SEASONS:
    season_df = fetch_season_boxscores(season)
    season_df.to_csv(f'boxscores_{season}.csv', index=False)
    print(season, len(season_df), 'rows')

## 4. Combine seasons -> `boxscores.csv`

This is the raw, full-stat dataset (points, rebounds, assists, 3PM, steals, blocks, turnovers, etc.) — consumed directly by `app_props.py`.

In [ ]:
boxscores = pd.concat(
    [pd.read_csv(f'boxscores_{season}.csv') for season in SEASONS],
    ignore_index=True
)
boxscores.to_csv('boxscores.csv', index=False)
boxscores.shape

## 5. Build the enriched dataset -> `boxscores_final.csv`

Adds team names/logos, opponent info, and matchup strings to every player-game row. This is the lightweight file `app.py` reads directly.

Each game's boxscore also contains two placeholder rows per team (`Player` == `"Team"` or `"Total"`, the team's aggregate line) — these get dropped here since they aren't real players.

In [ ]:
df = pd.read_csv('boxscores.csv')
teams = pd.read_csv('teams.csv')

df = df[~df['Player'].str.strip().str.upper().isin(['TEAM', 'TOTAL'])].copy()

# case-insensitive join keys
df['Team'] = df['Team'].str.upper()
df['hometeam'] = df['hometeam'].str.upper()
df['awayteam'] = df['awayteam'].str.upper()
teams['team.code'] = teams['team.code'].str.upper()
teams['team.name'] = teams['team.name'].str.upper()

# opponent = whichever of hometeam/awayteam isn't this player's own team
df['opponent_team'] = np.where(df['Team'] == df['hometeam'], df['awayteam'], df['hometeam'])

In [ ]:
# attach player's own team name/logo
df = df.merge(teams, how='left', left_on='Team', right_on='team.code')

# attach opponent's team name/logo
opponent_teams = teams.rename(columns={
    'team.code': 'opponent_team_code',
    'team.name': 'opponent_team_name',
    'team.imageUrl': 'opponent_team_imageUrl',
})
df = df.merge(opponent_teams, how='left', left_on='opponent_team', right_on='opponent_team_code')
df = df.drop(columns=['opponent_team'])

In [ ]:
df['Player'] = df['Player'].str.title()
df['team.name'] = df['team.name'].str.title()
df['opponent_team_name'] = df['opponent_team_name'].str.title()

df = df[[
    'date', 'Home', 'Player', 'Minutes', 'Points', 'TotalRebounds', 'Assistances',
    'team.code', 'team.name', 'team.imageUrl',
    'opponent_team_code', 'opponent_team_name', 'opponent_team_imageUrl'
]]

In [ ]:
df['matchup'] = df.apply(
    lambda row: f"{row['team.name']} vs {row['opponent_team_name']}"
    if row['Home'] == 1
    else f"{row['opponent_team_name']} @ {row['team.name']}",
    axis=1
)
df['matchup_axis'] = df.apply(
    lambda row: f"{row['team.code']} vs {row['opponent_team_code']}"
    if row['Home'] == 1
    else f"{row['opponent_team_code']} @ {row['team.code']}",
    axis=1
)
df = df.drop(columns=['Home'])

df.columns = [
    'Date', 'Player', 'Minutes', 'Points', 'Rebounds', 'Assists',
    'TeamCode', 'TeamName', 'TeamImageUrl',
    'OpponentTeamCode', 'OpponentTeamName', 'OpponentTeamImageUrl',
    'Matchup', 'MatchupAxis'
]

In [ ]:
df.to_csv('boxscores_final.csv', index=False)
df.head()

## Notes

- `boxscores_final.csv` only keeps Points/Rebounds/Assists plus team/matchup metadata. `app_props.py` merges it back with the fuller stat set in `boxscores.csv` (3PM, steals, blocks, turnovers, etc.) and the headshots in `players.csv` at read time, so there's no need to widen this file.
- To refresh existing seasons: re-run the whole notebook top to bottom.
- To add a new season: append it to `SEASONS` in the setup cell, re-run sections 2-3 (they fetch per season and won't touch other seasons' CSVs), then re-run sections 4-5.